# OFX

In [100]:
!pip install ofxtools
!pip install openpyxl

   ---------------------------------------- 0.0/250.0 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.0 kB ? eta -:--:--
   ---- ---------------------------------- 30.7/250.0 kB 435.7 kB/s eta 0:00:01
   -------------- ------------------------ 92.2/250.0 kB 871.5 kB/s eta 0:00:01
   ---------------------------------------  245.8/250.0 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 250.0/250.0 kB 1.5 MB/s eta 0:00:00


In [886]:
import os
import pandas as pd
import hashlib
from datetime import datetime
from ofxtools.Parser import OFXTree

#### Configurações

In [1]:
path_arquivos = './Arquivos/cartão/'

#### Funções

In [1158]:
def get_memo_info(memo):
    if 'DESC AUTOMATICO ANUIDADE' in memo:
        d = dict(
            tipo = 'anuidade_desconto',
            empresa = memo[:28].strip(),
            parc_numero = memo[28+6:28+6+2].strip(),
            pais = memo[-2:].strip(),
        )        
    elif 'ESTORNO  ANUIDADE' in memo or 'DESCONTO ANUIDADE' in memo:
        d = dict(
            tipo = 'anuidade_desconto',
            empresa = memo.strip(),
        )    
    elif 'DESCONTO' in memo:
        d = dict(
            tipo = 'anuidade_desconto',
            empresa = memo[:12].strip(),
            parc_numero = memo[12+8:12+8+2].strip(),
            parc_qtd = memo[12+8+2+1:12+8+2+1+2].strip(),
            pais = memo[-2:].strip(),
        )       
    elif 'ANUIDADE DIFERENCIADA' in memo:
        d = dict(
            tipo = 'anuidade',
            empresa = memo[:25].strip(),
            #parc_parc = memo[14:14+4],
            parc_numero = memo[25+6:25+6+2].strip(),
            parc_qtd = memo[25+6+2+1:25+6+2+1+2].strip(),
            pais = memo[-2:].strip(),
        ) 
    elif 'ANUIDADE' in memo:
        d = dict(
            tipo = 'anuidade',
            empresa = memo[:12].strip(),
            #parc_parc = memo[14:14+4],
            parc_numero = memo[12+8:12+8+2].strip(),
            parc_qtd = memo[12+8+2+1:12+8+2+1+2].strip(),
            pais = memo[-2:].strip(),
        )                    
    elif 'ANTEC ' in memo:
        d = dict(
            tipo = 'antecipacao',
            empresa = memo[7+2+1+2+1:7+2+1+2+2+1+18].strip(),
            #parc_parc = memo[14:14+4],
            parc_numero = memo[7:7+2].strip(),
            parc_qtd = memo[7+2+1:7+2+1+2].strip(),
            cidade = memo[7+2+1+2+2+1+18:].strip(),
        )   
    elif 'PGTO' in memo:
        d = dict(
            tipo = 'pagamento',
            agencia = memo[18:18+4].strip(),
        )   

    elif 'IOF' in memo:
        d = dict(
            tipo = 'iof',
        )

    elif 'DEBITOS PAGOS EM USS' in memo or 'DEBITOS NO EXTERIOR' in memo:
        d = dict(
            tipo = 'compras_exterior',
        )
    elif 'PARC ' in memo:
        # Regra Geral
        if len(memo) == 39:
            d = dict(
                tipo = 'parcelamento',
                empresa = memo[0:14].strip(),
                #parc_parc = memo[14:14+4],
                parc_numero = memo[14+4+1:14+4+1+2].strip(),
                parc_qtd = memo[14+4+1+2+1:14+4+1+2+1+2].strip(),
                cidade = memo[14+4+1+2+1+2:-2].strip(),
                pais   = memo[-2:].strip(),
            )        
        # Caso da M Martan
        if len(memo) == 37 and memo[-2:] == 'BR':
            d = dict(
                tipo = 'parcelamento',
                empresa = memo[0:12].strip(),
                #parc_parc = memo[12:12+4].strip(),
                parc_numero = memo[12+4+1:12+4+1+2].strip(),
                parc_qtd = memo[12+4+1+2+1:12+4+1+2+1+2].strip(),
                cidade = memo[12+4+1+2+1+2:-2].strip(),
                pais   = memo[-2:].strip(),
            )       
        else:
            d = dict(
                tipo = 'parcelamento',
                empresa = memo[0:14].strip(),
                #parc_parc = memo[14:14+4],
                parc_numero = memo[14+4+1:14+4+1+2].strip(),
                parc_qtd = memo[14+4+1+2+1:14+4+1+2+1+2].strip(),
                cidade = memo[14+4+1+2+1+2:].strip(),
            )    
    else:
        d = dict(
            tipo = 'a vista',
            empresa = memo[0:23].strip(),
            cidade = memo[23:-3].strip(),
            pais = memo[-3:].strip(),
        )

    return d

def get_hash_sha1(nome_arquivo):
    # Abrir o arquivo em modo binário para leitura
    with open(nome_arquivo, 'rb') as arquivo:
        # Criar um objeto hash SHA-1
        sha1 = hashlib.sha1()
        
        # Atualizar o objeto hash com os dados do arquivo em blocos
        while True:
            bloco = arquivo.read(4096)  # Ler em blocos de 4KB
            if not bloco:
                break
            sha1.update(bloco)
    
    # Retornar o hash SHA-1 em formato hexadecimal
    return sha1.hexdigest()

In [1141]:
'DESC AUTOMATICO ANUIDADE TIT-PARC 01/BR'[:28]
'DESC AUTOMATICO ANUIDADE TIT-PARC 01/BR'[28+6:28+6+2]
'DESC AUTOMATICO ANUIDADE TIT-PARC 01/BR'[28+6+2+1:28+6+2+1+2]

get_memo_info('DESC AUTOMATICO ANUIDADE TIT-PARC 01/BR')

{'tipo': 'anuidade_desconto',
 'empresa': 'DESC AUTOMATICO ANUIDADE TIT',
 'parc_numero': '01',
 'pais': 'BR'}

In [1172]:
# 2896 empresas
df_lancamentos[
    # (df_lancamentos['tipo'] == 'compras_exterior')
    # & (df_lancamentos['tipo'].isin(['anuidade']))
    (df_lancamentos['memo'].str.contains('DEBITOS PAGOS'))
    # & ~(df_lancamentos['memo'].str.contains('DESCONTO'))
    # & ~(df_lancamentos['memo'].str.contains('ANUIDADE'))
    # & (df_lancamentos['memo'].str[-2:] == 'BR')
    # & ~(df_lancamentos['parc_numero'].isin(['01', '04', '02', '03', '06', '07', '05', '10', '11', '08', '09', '12',]))
    # & (df_lancamentos['pais'] != 'BR') # 
    # & (df_lancamentos['memo_len'] < 39) # 
][['memo','tipo','empresa','cidade','pais','parc_numero','parc_qtd']].drop_duplicates()

,memo,tipo,empresa,cidade,pais,parc_numero,parc_qtd
5227,DEBITOS PAGOS EM USS NO PERIODO,compra_extrangeira,NaN,NaN,NaN,NaN,NaN


### Busca arquivos na pasta destino

In [679]:
contador = 0
lista_ofx = []

for pasta_atual, subpastas, arquivos in os.walk(path_arquivos):
        for arquivo in arquivos:
            caminho_completo = os.path.join(pasta_atual, arquivo)
            #print(caminho_completo)
            if arquivo.lower().split('.')[-1] == 'ofx':
                contador += 1
                lista_ofx.append(os.path.join(pasta_atual, arquivo))

print('Arquivos encontrados: {}'.format(contador))
# lista_ofx

Arquivos encontrados: 221


### Estudo de Tags do Arquivo

In [681]:
parser = OFXTree()
lista_element_tags = []
for nome_arquivo_ofx in lista_ofx:
    parser.parse(nome_arquivo_ofx)
    
    dict_tags = dict(nome=nome_arquivo_ofx)
    
    for elemento in parser._root.findall('.//*'):
        if elemento.tag not in dict_tags.keys():
            dict_tags[elemento.tag] = 1
        else:
            dict_tags[elemento.tag] += 1
    lista_element_tags.append(dict_tags)

pd.DataFrame(lista_element_tags).to_excel('tags.xlsx')

In [871]:

# Exemplo de uso
hash_sha1 = get_hash_sha1(lista_ofx[0])
print("SHA-1:", hash_sha1)



SHA-1: d12b2b73203f63264000bebfeb514e9051cfbaf2


In [1173]:
from ofxtools.Parser import OFXTree

parser = OFXTree()
lista_banktranlist = []
lista_ofx_cartao = []

for nome_arquivo in lista_ofx: #[0:5]:# ['./Arquivos/ofx/cartão/OUROCARD_MASTERCARD_BLACK-Próxima_Fatura.ofx']: #
    # Parser e converte o arquivo
    parser.parse(nome_arquivo)
    ofx = parser.convert()

    # Gera informações do arquivo
    hash_arquivo = get_hash_sha1(nome_arquivo)
    dt_criacao = datetime.fromtimestamp(os.path.getctime(nome_arquivo))

    # Informações para gerar identificação do arquivo
    acctid = ofx.statements[0].ccacctfrom.acctid
    periodo = ofx.statements[0].ledgerbal.dtasof.strftime('%Y-%m-%d')
    id_cartao = acctid[0:4] + '-' + acctid[-4:] + '-' + periodo

    # Informações gerais
    dict_ofx_cartao = dict (
        hash_arquivo = hash_arquivo,
        id_cartao = id_cartao,
        nome_arquivo = nome_arquivo,
        org = ofx.org,
        fid = ofx.fid,
        acctid = ofx.statements[0].ccacctfrom.acctid,
        balamt = float(ofx.statements[0].ledgerbal.balamt),
        dtasof = ofx.statements[0].ledgerbal.dtasof,
        dtstart = ofx.statements[0].banktranlist.dtstart,
        dtend = ofx.statements[0].banktranlist.dtend,
        dt_processamento = datetime.now(),
        dt_criacao = dt_criacao
    )

    lista_banktranlist_fatura = []
    nitem = 0
    
    for banktranlist in ofx.statements[0].banktranlist:
        # print(banktranlist)
        dict_banktranlist = dict(
            hash_arquivo = hash_arquivo,
            id_cartao = id_cartao,
            nitem = nitem,
            trntype = banktranlist.trntype,
            dtposted = banktranlist.dtposted,
            trnamt = float(banktranlist.trnamt),
            fitid = banktranlist.fitid,
            memo = banktranlist.memo,
            memo_len = len(banktranlist.memo),
            # **get_memo_info(banktranlist.memo),
        )

        # Acrescenta nitem
        nitem += 1 

        # Informações de compras no exterior
        try :
            dict_banktranlist['cursym'] = banktranlist.currency.cursym
            dict_banktranlist['currate'] = banktranlist.currency.currate
        except:
            pass

        # Decide o tipo de lançamento
        dict_banktranlist.update(get_memo_info(banktranlist.memo))

        # # Decide o tipo de lançamento
        # if 'PARC ' in banktranlist.memo:
        #     dict_banktranlist.update(get_memo_info(banktranlist.memo))
        # elif 'PGTO' in banktranlist.memo:
        #     dict_banktranlist.update(get_memo_info(banktranlist.memo))
        # elif 'IOF ' in banktranlist.memo:
        #     dict_banktranlist.update(get_memo_info(banktranlist.memo))
        # elif 'ANTEC ' in banktranlist.memo:
        #     dict_banktranlist.update(dict(tipo='antecipacao'))
        # elif 'DEBITOS PAGOS EM USS NO PERIODO' in banktranlist.memo:
        #     dict_banktranlist.update(dict(tipo='exterior'))
        # elif 'DEBITOS NO EXTERIOR CONVERT P/ R$' in banktranlist.memo:
        #     dict_banktranlist.update(dict(tipo='exterior'))
        # else:
        #     dict_banktranlist.update(get_memo_info(banktranlist.memo))

        if 'currate' in dict_banktranlist.keys():
            dict_banktranlist['tipo'] = 'compras_exterior'

        # Adiciona o lançamento a lista de lançamentos do arquivo
        lista_banktranlist_fatura.append(dict_banktranlist)

    
    # Quantidade
    dict_ofx_cartao['n_lancamentos'] = len([d['trnamt'] for d in lista_banktranlist_fatura])
    dict_ofx_cartao['n_pagamentos'] = len([d['trnamt'] for d in lista_banktranlist_fatura if d['tipo'] == 'pagamento'])
    dict_ofx_cartao['n_compras_nacionais'] = len([d['trnamt'] for d in lista_banktranlist_fatura if d['tipo'] not in ['pagamento','compra_extrangeira']])
    dict_ofx_cartao['n_compras_extrangeiras'] = len([d['trnamt'] for d in lista_banktranlist_fatura if d['tipo'] == 'compra_extrangeira'])
    # Soma
    dict_ofx_cartao['soma_pagamentos'] = round(sum([d['trnamt'] for d in lista_banktranlist_fatura if d['tipo'] in ['pagamento']]),2)
    dict_ofx_cartao['soma_compras_nacionais'] = round(sum([d['trnamt'] for d in lista_banktranlist_fatura if d['tipo'] not in ['pagamento','compra_extrangeira']]),2)
    dict_ofx_cartao['soma_compras_extrangeiras'] = round(sum([d['trnamt'] for d in lista_banktranlist_fatura if d['tipo'] in ['compra_extrangeira']]),2)

    # Visa dez/2016 houve pagamento antecipado e a soma de dolar foi zero
    if dict_ofx_cartao['n_compras_extrangeiras'] > 0 and abs(dict_ofx_cartao['soma_compras_extrangeiras']) > 0:
        #print(nome_arquivo_ofx,dict_ofx_cartao['n_compras_extrangeiras'],dict_ofx_cartao['balamt'],dict_ofx_cartao['soma_compras_nacionais'],dict_ofx_cartao['soma_compras_extrangeiras'])
        dict_ofx_cartao['cotacao'] = (dict_ofx_cartao['balamt'] - dict_ofx_cartao['soma_compras_nacionais']) / dict_ofx_cartao['soma_compras_extrangeiras']
    else:
        dict_ofx_cartao['cotacao'] = 5    

    # Transforma todos as compras em real
    for d in lista_banktranlist_fatura:
        if d['tipo'] == 'compra_extrangeira':
            d['valor_brl'] = round(dict_ofx_cartao['cotacao'] * d['trnamt'],2)
            # print('compra_extrangeira',dict_ofx_cartao['id_cartao'],dict_ofx_cartao['cotacao'],d['trnamt'],d['valor_brl'] )
        else:
            d['valor_brl'] = d['trnamt']
    dict_ofx_cartao['soma_valor_brl'] = round(sum([d['valor_brl'] for d in lista_banktranlist_fatura if d['tipo'] not in ['pagamento']]) ,2)   

    # Decide o tipo de arquivo: fatura_cartao ou fatura_cartao_provisoria
    if dict_ofx_cartao['n_pagamentos'] == 0 and dict_ofx_cartao['balamt'] == 0:
        dict_ofx_cartao['tpofx'] = 'fatura_cartao_provisoria'
    else:
        dict_ofx_cartao['tpofx'] = 'fatura_cartao'
        
    # Adiciona o cartão a lista
    lista_ofx_cartao.append(dict_ofx_cartao)
    lista_banktranlist = lista_banktranlist + lista_banktranlist_fatura

df_ofx_cartao = pd.DataFrame(lista_ofx_cartao)
df_lancamentos = pd.DataFrame(lista_banktranlist)

In [1048]:
# Nenhum cartão repetido até o momento
df_ofx_cartao.groupby('id_cartao').filter(lambda x: len(x) > 1)

,hash_arquivo,id_cartao,nome_arquivo,org,fid,acctid,balamt,dtasof,dtstart,dtend,...,n_lancamentos,n_pagamentos,n_compras_nacionais,n_compras_extrangeiras,soma_pagamentos,soma_compras_nacionais,soma_compras_extrangeiras,cotacao,soma_valor_brl,tpofx
152,36812d11aefdc2ae11d8f4af69b443be06be01e8,4984-5876-2016-12-10,./Arquivos/cartão/Visa\2016\OUROCARD_VISA_INFI...,Banco do Brasil,1,4984000000005876,4031.42,2016-12-10 00:00:00+00:00,2016-04-05 00:00:00+00:00,2016-11-24 00:00:00+00:00,...,88,1,84,3,3778.38,-4031.42,-0.0,5.0,-4031.42,fatura_cartao
163,73ab9cd5bd4d11aa450e618b31bcdd9020707758,4984-5876-2016-12-10,./Arquivos/cartão/Visa\2017\OUROCARD_VISA_INFI...,Banco do Brasil,1,4984000000005876,4031.42,2016-12-10 00:00:00+00:00,2016-04-05 00:00:00+00:00,2016-11-24 00:00:00+00:00,...,88,1,84,3,3778.38,-4031.42,-0.0,5.0,-4031.42,fatura_cartao


In [1145]:
df_ofx_cartao_selecao = df_ofx_cartao[
    (df_ofx_cartao['dtasof'] >= '2023-01-01')
    # & (df_ofx_cartao['n_compras_extrangeiras'] > 0)
]
df_lancamentos_selecao = df_lancamentos[df_lancamentos['hash_arquivo'].isin(df_ofx_cartao_selecao['hash_arquivo'])]

In [1146]:
df_consolidado = pd.pivot_table(
    df_lancamentos_selecao,
    index = 'id_cartao',
    columns = ['tipo'],
    values='valor_brl',
    aggfunc='sum',
    fill_value=0,
)
df_consolidado = df_consolidado.merge(df_ofx_cartao_selecao,on='id_cartao')

In [1051]:
# df_consolidado['compras'] = df_consolidado['a vista']+df_consolidado['antecipacao']+df_consolidado['iof']+df_consolidado['parcelamento']

df_erros = df_consolidado[
    (abs(df_consolidado['soma_valor_brl']) != abs(df_consolidado['balamt'])) 
    # & (df_consolidado_merged['compra_extrangeira'] == 0)
    # & (df_consolidado_merged['dtasof'] >= '2020-01-01')
    & (df_consolidado['tpofx'] != 'fatura_cartao_provisoria')
]#[['soma_valor_brl','balamt']]
# df_erros[['id_cartao','balamt','soma_pagamentos',
#        'soma_compras_nacionais', 'soma_compras_extrangeiras', 'cotacao',
#            'soma_valor_brl',]]

df_erros

,id_cartao,a vista,antecipacao,compra_extrangeira,iof,pagamento,parcelamento,hash_arquivo,nome_arquivo,org,...,n_lancamentos,n_pagamentos,n_compras_nacionais,n_compras_extrangeiras,soma_pagamentos,soma_compras_nacionais,soma_compras_extrangeiras,cotacao,soma_valor_brl,tpofx


In [1052]:
get_memo_info('MMARTAN     PARC 02/03 MACAPA      BR')
# get_memo_info('COMERCIAL LUI PARC 01/04 MACAPA      BR')
# get_memo_info('FARMACIA PREC PARC 01/06 FLORIANOPOLI')
# 'MMARTAN     PARC 02/03 MACAPA      BR'[12:]
# len('FARMACIA PREC PARC 01/06 FLORIANOPOLI')
# len('MMARTAN     PARC 02/03 MACAPA      BR')

{'tipo': 'parcelamento',
 'empresa': 'MMARTAN     ',
 'parc_numero': '02',
 'parc_nparc': '03',
 'cidade': ' MACAPA      ',
 'pais': 'BR'}

In [1068]:

        # elif 'ANTEC ' in banktranlist.memo:
        #     dict_banktranlist.update(dict(tipo='antecipacao'))
        # elif 'DEBITOS PAGOS EM USS NO PERIODO' in banktranlist.memo:
        #     dict_banktranlist.update(dict(tipo='exterior'))
        # elif 'DEBITOS NO EXTERIOR CONVERT P/ R$' in banktranlist.memo:
        #     dict_banktranlist.update(dict(tipo='exterior')) 
df_lancamentos[
    (df_lancamentos['tipo'] == 'antecipacao')
    # (df_lancamentos['memo'].str.contains('IOF'))
    & ~(df_lancamentos['memo'].str.contains('DESCONTO'))
    & ~(df_lancamentos['memo'].str.contains('ANUIDADE'))
    # & (df_lancamentos['memo'].str[-2:] == 'BR')
    # & ~(df_lancamentos['parc_numero'].isin(['01', '04', '02', '03', '06', '07', '05', '10', '11', '08', '09', '12',]))
    # & (df_lancamentos['pais'] != 'BR') # 
    # & (df_lancamentos['memo_len'] < 39) # 
]['memo'].unique() #[['memo','empresa','cidade','pais','parc_numero','parc_nparc']].drop_duplicates()

array(['ANTEC  02/12-EXTRA.COM           SAO PAU',
       'ANTEC  03/12-EXTRA.COM           SAO PAU',
       'ANTEC  04/12-EXTRA.COM           SAO PAU',
       'ANTEC  05/12-EXTRA.COM           SAO PAU',
       'ANTEC  06/12-EXTRA.COM           SAO PAU',
       'ANTEC  07/12-EXTRA.COM           SAO PAU',
       'ANTEC  08/12-EXTRA.COM           SAO PAU',
       'ANTEC  09/12-EXTRA.COM           SAO PAU',
       'ANTEC  10/12-EXTRA.COM           SAO PAU',
       'ANTEC  11/12-EXTRA.COM           SAO PAU',
       'ANTEC  12/12-EXTRA.COM           SAO PAU',
       'MercPag GPCAVALCANTEC  OSASCO        BR',
       'ANTEC  02/03-PETLOVE Order10     SAO PAU',
       'ANTEC  03/03-PETLOVE Order10     SAO PAU',
       'ANTEC  02/08-MercPag PERSIAN     OSASCO',
       'ANTEC  03/08-MercPag PERSIAN     OSASCO',
       'ANTEC  04/08-MercPag PERSIAN     OSASCO',
       'ANTEC  05/08-MercPag PERSIAN     OSASCO',
       'ANTEC  06/08-MercPag PERSIAN     OSASCO',
       'ANTEC  07/08-MercPag PERSIAN 

In [1057]:
# 2889 existentes
# 708 são de 2023 pra ca
df_lancamentos[
    (df_lancamentos['tipo'] != 'pagamento')
    & ~(df_lancamentos['memo'].str.contains('DESCONTO'))
    & ~(df_lancamentos['memo'].str.contains('ANUIDADE'))
    # & (df_lancamentos['memo'].str[-2:] == 'BR')
    # & ~(df_lancamentos['parc_numero'].isin(['01', '04', '02', '03', '06', '07', '05', '10', '11', '08', '09', '12',]))
    # & (df_lancamentos['pais'] != 'BR') # 
    # & (df_lancamentos['memo_len'] < 39) # 
]#['empresa'].nunique() #[['memo','empresa','cidade','pais','parc_numero','parc_nparc']].drop_duplicates()

,hash_arquivo,id_cartao,nitem,trntype,dtposted,trnamt,fitid,memo,memo_len,tipo,empresa,cidade,pais,valor_brl,parc_numero,parc_nparc,agencia,cursym,currate
0,d12b2b73203f63264000bebfeb514e9051cfbaf2,6516-9883-2018-04-10,0,PAYMENT,2018-03-20 00:00:00+00:00,-9.00,2018032065160000000098830000000000,SORVETERIA STA CLARA MACAPA BR,39,a vista,SORVETERIA STA CLARA,MACAPA,BR,-9.00,NaN,NaN,NaN,NaN,NaN
1,d12b2b73203f63264000bebfeb514e9051cfbaf2,6516-9883-2018-04-10,1,PAYMENT,2018-03-23 00:00:00+00:00,-77.40,2018032365160000000098830000000001,BOSS BURGER MACAPA BR,39,a vista,BOSS BURGER,MACAPA,BR,-77.40,NaN,NaN,NaN,NaN,NaN
2,d12b2b73203f63264000bebfeb514e9051cfbaf2,6516-9883-2018-04-10,2,PAYMENT,2018-03-24 00:00:00+00:00,-58.50,2018032465160000000098830000000002,JUICE TRUCK MACAPA BR,39,a vista,JUICE TRUCK,MACAPA,BR,-58.50,NaN,NaN,NaN,NaN,NaN
3,d12b2b73203f63264000bebfeb514e9051cfbaf2,6516-9883-2018-04-10,3,PAYMENT,2018-03-25 00:00:00+00:00,-60.00,2018032565160000000098830000000003,SOGOOD ALIMENTOS LTDA MACAPA BR,39,a vista,SOGOOD ALIMENTOS LTDA,MACAPA,BR,-60.00,NaN,NaN,NaN,NaN,NaN
4,d12b2b73203f63264000bebfeb514e9051cfbaf2,6516-9883-2018-04-10,4,PAYMENT,2018-03-20 00:00:00+00:00,-9.00,2018032065160000000098830000000004,PANIFICADORA CAROL MACAPA BR,39,a vista,PANIFICADORA CAROL,MACAPA,BR,-9.00,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13595,4835c01edf60556108082a1ab4e808e46e202d94,4984-0550-2024-05-06,2,PAYMENT,2024-05-01 00:00:00+00:00,-68.78,2024050149840000000005500000000002,SUPERMERCADO FORTALEZ MACAPA,29,a vista,SUPERMERCADO FORTALEZ,MAC,APA,-68.78,NaN,NaN,NaN,NaN,NaN
13596,4835c01edf60556108082a1ab4e808e46e202d94,4984-0550-2024-05-06,3,PAYMENT,2024-05-02 00:00:00+00:00,-203.86,2024050249840000000005500000000003,DAVID PENHA SILVA EIRE MACAPA,29,a vista,DAVID PENHA SILVA EIRE,MAC,APA,-203.86,NaN,NaN,NaN,NaN,NaN
13597,4835c01edf60556108082a1ab4e808e46e202d94,4984-0550-2024-05-06,4,PAYMENT,2024-05-05 00:00:00+00:00,-177.65,2024050549840000000005500000000004,JOAO DO CAMARAO MACAPA,29,a vista,JOAO DO CAMARAO,MAC,APA,-177.65,NaN,NaN,NaN,NaN,NaN
13598,4835c01edf60556108082a1ab4e808e46e202d94,4984-0550-2024-05-06,5,PAYMENT,2024-05-06 00:00:00+00:00,-40.00,2024050649840000000005500000000005,DISTRIBUIDORA TOP MACAPA,29,a vista,DISTRIBUIDORA TOP,MAC,APA,-40.00,NaN,NaN,NaN,NaN,NaN


In [431]:
'AMERICANAS.CO PARC 02/07 RIO DE JANEIBR',

'BRA'

In [ ]:
ANUIDADE DIFER

In [ ]:
'FARMACIA PREC PARC 01/06 FLORIANOPOLI'

In [378]:

# get_memo_parc('VIAC PARC 03/03 JEQUIE BR')
df[['memo']].assign(**get_memo_info(df['memo']))

0                 DEBITOS PAGOS EM USS NO PERIODO
1               DEBITOS NO EXTERIOR CONVERT P/ R$
2        MERCPAGO               OSASCO        BRA
3        MERCPAGO               OSASCO        BRA
4        MERCPAGO               OSASCO        BRA
                           ...                   
34349     WL ATACADISTA PARC 02/04 BRASILIA    BR
34350     VIACAO CIDADE PARC 03/03 JEQUIE      BR
34351     VIACAO CIDADE PARC 02/02 JEQUIE      BR
34352     TAM SITE      PARC 04/04 SAO PAULO   BR
34353     MAGAZINE LUIZ PARC 03/12 CAMPINAS    BR
Name: memo, Length: 34354, dtype: object


,memo,tipo
0,DEBITOS PAGOS EM USS NO PERIODO,outros
1,DEBITOS NO EXTERIOR CONVERT P/ R$,outros
2,MERCPAGO OSASCO BRA,outros
3,MERCPAGO OSASCO BRA,outros
4,MERCPAGO OSASCO BRA,outros
...,...,...
34349,WL ATACADISTA PARC 02/04 BRASILIA BR,outros
34350,VIACAO CIDADE PARC 03/03 JEQUIE BR,outros
34351,VIACAO CIDADE PARC 02/02 JEQUIE BR,outros
34352,TAM SITE PARC 04/04 SAO PAULO BR,outros


In [339]:
df.assign(**dict(teste=1,teste2=2,teste3=df['memo']))


,nome_arquivo,trntype,dtposted,trnamt,fitid,memo,memo_len,cursym,currate,parc_pos,memo_0,memo_1,memo_2,memo_3,memo_4,teste,teste2,teste3
0,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-11 00:00:00+00:00,603.10,2018011155220000000060050000000000,DEBITOS PAGOS EM USS NO PERIODO,31,USD,0.0000,NaN,DEBITOS PAGOS,1,2,PERIO,DO,1,2,DEBITOS PAGOS EM USS NO PERIODO
1,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,PAYMENT,2018-01-11 00:00:00+00:00,-603.10,2018011155220000000060050000000001,DEBITOS NO EXTERIOR CONVERT P/ R$,33,USD,0.0000,NaN,DEBITOS NO EXT,1,2,ERT P/,R$,1,2,DEBITOS NO EXTERIOR CONVERT P/ R$
2,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-08 00:00:00+00:00,7508.07,2018010855220000000060050000000002,MERCPAGO OSASCO BRA,40,NaN,NaN,NaN,MERCPAGO,1,2,SASCO B,RA,1,2,MERCPAGO OSASCO BRA
3,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-08 00:00:00+00:00,6999.96,2018010855220000000060050000000003,MERCPAGO OSASCO BRA,40,NaN,NaN,NaN,MERCPAGO,1,2,SASCO B,RA,1,2,MERCPAGO OSASCO BRA
4,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-10 00:00:00+00:00,1600.15,2018011055220000000060050000000004,MERCPAGO OSASCO BRA,40,NaN,NaN,NaN,MERCPAGO,1,2,SASCO B,RA,1,2,MERCPAGO OSASCO BRA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34349,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-12-21 00:00:00+00:00,-425.00,2017122155220000000060050000000173,WL ATACADISTA PARC 02/04 BRASILIA BR,39,NaN,NaN,14.0,WL ATACADISTA,1,2,BRASILIA,BR,1,2,WL ATACADISTA PARC 02/04 BRASILIA BR
34350,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-11-22 00:00:00+00:00,-40.66,2017112255220000000060050000000174,VIACAO CIDADE PARC 03/03 JEQUIE BR,39,NaN,NaN,14.0,VIACAO CIDADE,1,2,JEQUIE,BR,1,2,VIACAO CIDADE PARC 03/03 JEQUIE BR
34351,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-12-22 00:00:00+00:00,-68.27,2017122255220000000060050000000175,VIACAO CIDADE PARC 02/02 JEQUIE BR,39,NaN,NaN,14.0,VIACAO CIDADE,1,2,JEQUIE,BR,1,2,VIACAO CIDADE PARC 02/02 JEQUIE BR
34352,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-10-20 00:00:00+00:00,-37.50,2017102055220000000060050000000176,TAM SITE PARC 04/04 SAO PAULO BR,39,NaN,NaN,14.0,TAM SITE,1,2,SAO PAULO,BR,1,2,TAM SITE PARC 04/04 SAO PAULO BR


In [335]:
df

,nome_arquivo,trntype,dtposted,trnamt,fitid,memo,memo_len,cursym,currate,parc_pos,memo_0,memo_1,memo_2,memo_3,memo_4
0,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-11 00:00:00+00:00,603.10,2018011155220000000060050000000000,DEBITOS PAGOS EM USS NO PERIODO,31,USD,0.0000,NaN,DEBITOS PAGOS,1,2,PERIO,DO
1,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,PAYMENT,2018-01-11 00:00:00+00:00,-603.10,2018011155220000000060050000000001,DEBITOS NO EXTERIOR CONVERT P/ R$,33,USD,0.0000,NaN,DEBITOS NO EXT,1,2,ERT P/,R$
2,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-08 00:00:00+00:00,7508.07,2018010855220000000060050000000002,MERCPAGO OSASCO BRA,40,NaN,NaN,NaN,MERCPAGO,1,2,SASCO B,RA
3,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-08 00:00:00+00:00,6999.96,2018010855220000000060050000000003,MERCPAGO OSASCO BRA,40,NaN,NaN,NaN,MERCPAGO,1,2,SASCO B,RA
4,./Arquivos/ofx/cartão/OUROCARD_ELO_NANQUIM-Abr...,CREDIT,2018-01-10 00:00:00+00:00,1600.15,2018011055220000000060050000000004,MERCPAGO OSASCO BRA,40,NaN,NaN,NaN,MERCPAGO,1,2,SASCO B,RA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34349,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-12-21 00:00:00+00:00,-425.00,2017122155220000000060050000000173,WL ATACADISTA PARC 02/04 BRASILIA BR,39,NaN,NaN,14.0,WL ATACADISTA,1,2,BRASILIA,BR
34350,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-11-22 00:00:00+00:00,-40.66,2017112255220000000060050000000174,VIACAO CIDADE PARC 03/03 JEQUIE BR,39,NaN,NaN,14.0,VIACAO CIDADE,1,2,JEQUIE,BR
34351,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-12-22 00:00:00+00:00,-68.27,2017122255220000000060050000000175,VIACAO CIDADE PARC 02/02 JEQUIE BR,39,NaN,NaN,14.0,VIACAO CIDADE,1,2,JEQUIE,BR
34352,./Arquivos/ofx/cartão/Visa\2019\OUROCARD_VISA_...,PAYMENT,2017-10-20 00:00:00+00:00,-37.50,2017102055220000000060050000000176,TAM SITE PARC 04/04 SAO PAULO BR,39,NaN,NaN,14.0,TAM SITE,1,2,SAO PAULO,BR


In [328]:
df['memo'].apply(lambda x:get_memo_info(x))

0                 [DEBITOS PAGOS , S , O , PERIO, DO]
1               [DEBITOS NO EXT,  C, NV, ERT P/ , R$]
2        [MERCPAGO      ,   ,  O, SASCO        B, RA]
3        [MERCPAGO      ,   ,  O, SASCO        B, RA]
4        [MERCPAGO      ,   ,  O, SASCO        B, RA]
                             ...                     
34349     [WL ATACADISTA , 02, 04,  BRASILIA    , BR]
34350     [VIACAO CIDADE , 03, 03,  JEQUIE      , BR]
34351     [VIACAO CIDADE , 02, 02,  JEQUIE      , BR]
34352     [TAM SITE      , 04, 04,  SAO PAULO   , BR]
34353     [MAGAZINE LUIZ , 03, 12,  CAMPINAS    , BR]
Name: memo, Length: 34354, dtype: object

In [329]:
df[[]] = df['memo'].apply(lambda x:get_memo_info(x))

ValueError: Columns must be same length as key

In [1025]:
get_memo_info('DOMESTILAR PARC 05/05 MACAPA BR')

{'tipo': 'parcelamento',
 'empresa': 'DOMESTILAR PAR',
 'parc_numero': '05',
 'parc_nparc': 'MA',
 'cidade': 'CAPA BR'}

In [287]:
# 193 arquivos
df['parc_pos'] = df[df['memo'].str.contains('PARC ')]['memo'].str.find('PARC')
df['parc_pos'].value_counts()

get_memo_parc(
# PARC sempre no 14

df['parc_empresa'] = df['memo'].str[0:14]
df['pard_'] = df['memo'].str[14:14+4]
df['memo_2'] = df['memo'].str[14+4+1:14+4+1+2]
df['memo_3'] = df['memo'].str[14+4+1+2+1:14+4+1+2+1+2]
df['memo_3'] = df['memo'].str[14+4+1+2+1+2:-2]
df['memo_4'] = df['memo'].str[-2:]
df[df['memo'].str.contains('PARC ')][['memo','memo_0','memo_1','memo_2','memo_3','memo_4']].drop_duplicates()

,memo,memo_0,memo_1,memo_2,memo_3,memo_4
81,DOMESTILAR PARC 05/05 MACAPA BR,DOMESTILAR,PARC,05,MACAPA,BR
82,CROSSFIT M96 PARC 03/03 Macapá BR,CROSSFIT M96,PARC,03,Macapá,BR
83,NOBREGA PARC 05/06 FLORIANOPOLIBR,NOBREGA,PARC,05,FLORIANOPOLI,BR
84,CORELLO AROUC PARC 01/03 BRASILIA BR,CORELLO AROUC,PARC,01,BRASILIA,BR
85,LB PARK SHOPP PARC 01/04 BRASILIA BR,LB PARK SHOPP,PARC,01,BRASILIA,BR
86,LIVRARIA CULT PARC 01/03 BRASILIA BR,LIVRARIA CULT,PARC,01,BRASILIA,BR
87,TAM SITE PARC 02/04 SAO PAULO BR,TAM SITE,PARC,02,SAO PAULO,BR
88,ARAMIS PARKSH PARC 01/05 GUARA BR,ARAMIS PARKSH,PARC,01,GUARA,BR
89,AMAZON.COM.BR PARC 01/02 SAO PAULO BR,AMAZON.COM.BR,PARC,01,SAO PAULO,BR
90,MERCPAGO*MEMO PARC 01/12 OSASCO BR,MERCPAGO*MEMO,PARC,01,OSASCO,BR


In [222]:
print(ofx)
print(ofx.org)
print(ofx.fid)
# print(ofx.statements)
# ccacctfrom
# banktranlist
# ledgerbal
print(ofx.statements[0].ccacctfrom.acctid)
print(ofx.statements[0].ledgerbal.balamt)
print(ofx.statements[0].ledgerbal.dtasof)

print(ofx.statements[0].banktranlist.dtstart)
print(ofx.statements[0].banktranlist.dtend)
print(ofx.statements[0].banktranlist[0])

lista_banktranlist = []
for banktranlist in ofx.statements[0].banktranlist:
    # print(banktranlist)
    dict_banktranlist = dict(
        trntype = banktranlist.trntype,
        dtposted = banktranlist.dtposted,
        trnamt = banktranlist.trnamt,
        fitid = banktranlist.fitid,
        memo = banktranlist.memo,
        len_memo = len(banktranlist.memo)
    )
    try :
        dict_banktranlist['cursym'] = banktranlist.currency.cursym
        dict_banktranlist['currate'] = banktranlist.currency.currate
    except:
        pass

    lista_banktranlist.append(dict_banktranlist)

# lista_banktranlist
# ofx.statements[0].banktranlist[0].find('CURRENCY')
# print(ofx.statements[0].banktranlist[0].CURRENCY)

# for l in ofx.statements[0].banktranlist:
#     print(l)

# dir(ofx)
df = pd.DataFrame(lista_banktranlist)
df
df[df['trntype'] == 'PAYMENT'].trnamt.sum()

<OFX fid='1' org='Banco do Brasil' len(statements)=1 len(securities)=0>
Banco do Brasil
1
5522000000006005
-13866.16
2018-02-10 00:00:00+00:00
2017-08-12 00:00:00+00:00
2018-01-30 00:00:00+00:00
<STMTTRN(trntype='CREDIT', dtposted=datetime.datetime(2018, 1, 11, 0, 0, tzinfo=<UTC>), trnamt=Decimal('603.10'), fitid='2018011155220000000060050000000000', memo='DEBITOS PAGOS EM USS NO PERIODO', currency=<CURRENCY(currate=Decimal('0.0000'), cursym='USD')>)>


Decimal('-29148.52')

In [223]:
# (6912.53 - 6662.50+61.56)/61.56
13866.16 - 29148.52

-15282.36

In [228]:
df['memo_len2'] = df['memo'].str.len()
df

,trntype,dtposted,trnamt,fitid,memo,len_memo,cursym,currate,memo_len2
0,CREDIT,2018-01-11 00:00:00+00:00,603.10,2018011155220000000060050000000000,DEBITOS PAGOS EM USS NO PERIODO,31,USD,0.0000,31
1,PAYMENT,2018-01-11 00:00:00+00:00,-603.10,2018011155220000000060050000000001,DEBITOS NO EXTERIOR CONVERT P/ R$,33,USD,0.0000,33
2,CREDIT,2018-01-08 00:00:00+00:00,7508.07,2018010855220000000060050000000002,MERCPAGO OSASCO BRA,40,NaN,NaN,40
3,CREDIT,2018-01-08 00:00:00+00:00,6999.96,2018010855220000000060050000000003,MERCPAGO OSASCO BRA,40,NaN,NaN,40
4,CREDIT,2018-01-10 00:00:00+00:00,1600.15,2018011055220000000060050000000004,MERCPAGO OSASCO BRA,40,NaN,NaN,40
...,...,...,...,...,...,...,...,...,...
173,PAYMENT,2017-12-21 00:00:00+00:00,-425.00,2017122155220000000060050000000173,WL ATACADISTA PARC 02/04 BRASILIA BR,39,NaN,NaN,39
174,PAYMENT,2017-11-22 00:00:00+00:00,-40.66,2017112255220000000060050000000174,VIACAO CIDADE PARC 03/03 JEQUIE BR,39,NaN,NaN,39
175,PAYMENT,2017-12-22 00:00:00+00:00,-68.27,2017122255220000000060050000000175,VIACAO CIDADE PARC 02/02 JEQUIE BR,39,NaN,NaN,39
176,PAYMENT,2017-10-20 00:00:00+00:00,-37.50,2017102055220000000060050000000176,TAM SITE PARC 04/04 SAO PAULO BR,39,NaN,NaN,39


In [224]:
df[(df['trntype'] == 'CREDIT') & (df['cursym'].isna())]#['trnamt']#.sum()

,trntype,dtposted,trnamt,fitid,memo,len_memo,cursym,currate
2,CREDIT,2018-01-08 00:00:00+00:00,7508.07,2018010855220000000060050000000002,MERCPAGO OSASCO BRA,40,NaN,NaN
3,CREDIT,2018-01-08 00:00:00+00:00,6999.96,2018010855220000000060050000000003,MERCPAGO OSASCO BRA,40,NaN,NaN
4,CREDIT,2018-01-10 00:00:00+00:00,1600.15,2018011055220000000060050000000004,MERCPAGO OSASCO BRA,40,NaN,NaN
5,CREDIT,2018-01-10 00:00:00+00:00,12481.08,2018011055220000000060050000000005,PGTO DEBITO CONTA 5929 000004008 200211,40,NaN,NaN


In [212]:
29192.36 - 603.10 - 12481.08

16108.180000000002

In [216]:
df[~df['cursym'].isna()]

,trntype,dtposted,trnamt,fitid,memo,cursym,currate
0,CREDIT,2018-01-11 00:00:00+00:00,603.10,2018011155220000000060050000000000,DEBITOS PAGOS EM USS NO PERIODO,USD,0.0000
1,PAYMENT,2018-01-11 00:00:00+00:00,-603.10,2018011155220000000060050000000001,DEBITOS NO EXTERIOR CONVERT P/ R$,USD,0.0000
43,PAYMENT,2018-01-05 00:00:00+00:00,-49.20,2018010555220000000060050000000043,WWW.ALIEXPRESS.COM LONDON GBR,USD,0.0000
44,PAYMENT,2018-01-08 00:00:00+00:00,-3.14,2018010855220000000060050000000044,IOF - COMPRA NO EXTERIOR,USD,0.0000
46,PAYMENT,2018-01-05 00:00:00+00:00,-151.51,2018010555220000000060050000000046,WWW.ALIEXPRESS.COM LONDON GBR,USD,0.0000
47,PAYMENT,2018-01-08 00:00:00+00:00,-9.67,2018010855220000000060050000000047,IOF - COMPRA NO EXTERIOR,USD,0.0000
51,PAYMENT,2018-01-08 00:00:00+00:00,-49.20,2018010855220000000060050000000051,ALIEXPRESS LUXEMBOURG LUX,USD,0.0000
52,PAYMENT,2018-01-10 00:00:00+00:00,-3.14,2018011055220000000060050000000052,IOF - COMPRA NO EXTERIOR,USD,0.0000
53,PAYMENT,2018-01-08 00:00:00+00:00,-317.01,2018010855220000000060050000000053,ALIEXPRESS LUXEMBOURG LUX,USD,0.0000
54,PAYMENT,2018-01-10 00:00:00+00:00,-20.23,2018011055220000000060050000000054,IOF - COMPRA NO EXTERIOR,USD,0.0000
